In [1]:
import sys
import os


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
print(sys.path[-1])

/Users/vashqu/Desktop/MLtopKin/MLtopKin


In [2]:
import pandas as pd
import numpy as np
from src.utils import (
    prep_kinase_data_list,
    create_kinase_dataset,
    extract_features_for_ml
)

In [3]:
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR  = os.path.join(BASE_DIR, 'data')
METADATA_DIR = os.path.join(BASE_DIR, 'metadata')
FEATURES_DIR = os.path.join(BASE_DIR, 'features')

os.makedirs(FEATURES_DIR, exist_ok=True)

print(BASE_DIR)
print(DATA_DIR)
print(METADATA_DIR)
print(FEATURES_DIR)

/Users/vashqu/Desktop/MLtopKin/MLtopKin
/Users/vashqu/Desktop/MLtopKin/MLtopKin/data
/Users/vashqu/Desktop/MLtopKin/MLtopKin/metadata
/Users/vashqu/Desktop/MLtopKin/MLtopKin/features


In [4]:
# Only the chains selected in the 1_download_clean_pdbs notebook
df = pd.read_csv(
    os.path.join(METADATA_DIR, 'selected_chains.tab'), sep='\t'
)
df.head()

,Organism,Group,Gene,UniprotID,PDB,Method,Resolution,Rfac,FreeRfac,DomainBoundary,...,DihedralLabel,ActivityLabel,Chelix-Saltbr,ActLoop,Ligand,LigandType,DFG_Phe,Phosphorylation,Mutations,ProteinName
0,Arabidopsis thaliana,TKL,CTR1,CTR1_ARATH,3PPZB,XRAY,2.990,0.204,0.256,551-809,...,BLAminus,Active,in-in,in-in,No_ligand,No_ligand,695,X704 X707 X710,NaN,Serine/threonine-protein kinase CTR1
1,Arabidopsis thaliana,TKL,CTR1,CTR1_ARATH,3P86B,XRAY,2.496,0.197,0.232,551-809,...,NaN,Inactive,in-out,none-out,No_ligand,No_ligand,695,NaN,D676N,Serine/threonine-protein kinase CTR1
2,Bos taurus,AGC,GRK1,GRK1_BOVIN,4L9IA,XRAY,2.320,0.188,0.231,187-452,...,BLAminus,Inactive,in-out,in-in,8PR:601,Type1,333,NaN,NaN,Rhodopsin kinase GRK1
3,Bos taurus,AGC,GRK1,GRK1_BOVIN,4WBOA,XRAY,2.810,0.243,0.267,187-452,...,BLAminus,Inactive,in-out,in-in,ANW:601,Type1,333,NaN,NaN,Rhodopsin kinase GRK1
4,Bos taurus,AGC,GRK1,GRK1_BOVIN,4WBOB,XRAY,2.810,0.243,0.267,187-452,...,BLAminus,Inactive,in-out,in-in,ANW:601,Type1,333,NaN,NaN,Rhodopsin kinase GRK1


In [5]:
# Prepare kinase data list from directory structure
kinase_data_list = prep_kinase_data_list(
    in_path=DATA_DIR,
    df=df,
    gene_col='Gene',        # column name containing individual gene names
    group_col='Group',      # column name containing gene groups/families
    pdb_col='PDB',          # column name containing PDB IDs
    domain_col='DomainBoundary'  # use only the kinase domain of each chain
)

# Fixed image ranges (in squared Angstrom, the alpha complex filtration
# values are squared radii). They are set beforehand and not fitted on the
# dataset, so no information from test structures enters the features.

# Create kinases with parallel processing
kinases = create_kinase_dataset(
    kinase_data_list,
    max_workers=None,          # Use 8 parallel workers
    C_alpha_only=False,      # Use all atoms
    pixel_size=0.5,         # Higher resolution persistence images
    birth_range=(0, 16),
    pers_range=(0, 4)
)

# Extract features for machine learning
X, y, metadata = extract_features_for_ml(kinases, include_dims=[0, 1, 2])

Scanning directory: /Users/vashqu/Desktop/MLtopKin/MLtopKin/data
DataFrame shape: (2166, 25)



=== Processing Summary ===
Total PDB files found: 2166
Successfully processed: 2166
Skipped - insufficient path depth: 0
Skipped - invalid activity label: 0
Skipped - parsing errors: 0
Skipped - not found in DataFrame: 0
Skipped - duplicates: 0
Skipped - file access errors: 0

=== Activity Distribution ===
Active: 1083
Inactive: 1083


Processing kinases:   0%|          | 0/2166 [00:00<?, ?it/s]

Processing kinases:   0%|          | 1/2166 [00:01<58:43,  1.63s/it]

Processing kinases:   0%|          | 9/2166 [00:01<06:08,  5.86it/s]

Processing kinases:   1%|          | 16/2166 [00:02<03:08, 11.38it/s]

... (652 outputs left out)

Processing kinases: 100%|█████████▉| 2162/2166 [01:52<00:00, 16.79it/s]

Processing kinases: 100%|██████████| 2166/2166 [01:52<00:00, 19.99it/s]

Processing kinases: 100%|██████████| 2166/2166 [01:52<00:00, 19.27it/s]

Failed to create kinase for MAPK12_1CM8_B: No valid atoms found for /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/MAPK12/Active/1CM8_chainB.cif
Failed to create kinase for HASPIN_6G36_A: No valid atoms found for /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/HASPIN/Active/6G36_chainA.cif
Failed to create kinase for LYN_3A4O_X: Only 28 of the 253 domain residues found for /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/LYN/Inactive/3A4O_chainX.cif


In [6]:
np.savez_compressed(
    os.path.join(FEATURES_DIR, 'kincore.npz'), 
    X=X, y=y, metadata=np.array(metadata, dtype=object)
)

In [7]:
# To load:
data = np.load(os.path.join(FEATURES_DIR, 'kincore.npz'), allow_pickle=True)
X, y, metadata = data['X'], data['y'], data['metadata']

In [8]:
X

array([[1.87211155e-02, 1.90216357e-02, 1.51692799e-02, ...,
        5.91132962e-06, 2.66938591e-06, 9.61064478e-07],
       [1.87131400e-02, 1.90078441e-02, 1.51529840e-02, ...,
        4.35961600e-05, 2.24658942e-05, 9.38019213e-06],
       [1.86124637e-02, 1.88534429e-02, 1.49743639e-02, ...,
        1.93167229e-05, 2.25945798e-05, 2.39423530e-05],
       ...,
       [1.86266152e-02, 1.88873177e-02, 1.50291638e-02, ...,
        6.68225539e-05, 5.22024845e-05, 3.25348972e-05],
       [1.86428866e-02, 1.88899246e-02, 1.50072299e-02, ...,
        2.31989744e-05, 1.38715059e-05, 8.78582434e-06],
       [1.86466598e-02, 1.88929482e-02, 1.50087037e-02, ...,
        2.87236445e-05, 1.87313742e-05, 1.08200093e-05]],
      shape=(2163, 520))

In [9]:
y

array([1, 0, 1, ..., 1, 0, 0], shape=(2163,))

In [10]:
metadata

array([{'name': 'AAK1_8GMD_A', 'activity': 'active', 'pdb_id': '8GMD', 'chain': 'A', 'gene': 'AAK1', 'group': 'OTHER', 'n_atoms': 1988, 'sasa': 12573.080911724095, 'normalized_sasa': 0.4142893712837648, 'molecular_weight': 30348.5481},
       {'name': 'AAK1_7LVI_A', 'activity': 'inactive', 'pdb_id': '7LVI', 'chain': 'A', 'gene': 'AAK1', 'group': 'OTHER', 'n_atoms': 2048, 'sasa': 12395.672795609727, 'normalized_sasa': 0.40644476418581654, 'molecular_weight': 30497.804099999998},
       {'name': 'ABL1_2F4J_A', 'activity': 'active', 'pdb_id': '2F4J', 'chain': 'A', 'gene': 'ABL1', 'group': 'TYR', 'n_atoms': 2055, 'sasa': 13193.38862432337, 'normalized_sasa': 0.45140521607341755, 'molecular_weight': 29227.373},
       ...,
       {'name': 'WNK3_5O26_B', 'activity': 'active', 'pdb_id': '5O26', 'chain': 'B', 'gene': 'WNK3', 'group': 'OTHER', 'n_atoms': 1983, 'sasa': 12639.808067914246, 'normalized_sasa': 0.4394006120294872, 'molecular_weight': 28766.022900000004},
       {'name': 'WNK3_5O1V_A

In [11]:
# ========================================== #
# Same as above, but only carbon alpha atoms #
# ========================================== #

# Create kinases with parallel processing
kinases = create_kinase_dataset(
    kinase_data_list,
    max_workers=None,          # Use 8 parallel workers
    C_alpha_only=True,      # Use only C-alpha atoms for faster computation
    pixel_size=1.0,         # C-alpha features live on larger scales
    birth_range=(0, 48),
    pers_range=(0, 16)
)

# Extract features for machine learning
X, y, metadata = extract_features_for_ml(kinases, include_dims=[0, 1, 2])

np.savez_compressed(
    os.path.join(FEATURES_DIR, 'kincore_Calphaonly.npz'), 
    X=X, y=y, metadata=np.array(metadata, dtype=object)
)

Processing kinases:   0%|          | 0/2166 [00:00<?, ?it/s]

Processing kinases:   0%|          | 1/2166 [00:02<1:22:04,  2.27s/it]

Processing kinases:   0%|          | 8/2166 [00:02<07:57,  4.52it/s]  

Processing kinases:   1%|          | 12/2166 [00:02<05:12,  6.89it/s]

Processing kinases:   1%|          | 17/2166 [00:02<03:25, 10.48it/s]

Processing kinases:   1%|          | 21/2166 [00:02<02:41, 13.30it/s]

... (420 outputs left out)

Processing kinases: 100%|█████████▉| 2161/2166 [00:59<00:00, 42.88it/s]

Processing kinases: 100%|██████████| 2166/2166 [00:59<00:00, 42.35it/s]

Processing kinases: 100%|██████████| 2166/2166 [00:59<00:00, 36.13it/s]

Failed to create kinase for MAPK12_1CM8_B: No valid atoms found for /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/MAPK12/Active/1CM8_chainB.cif
Failed to create kinase for LYN_3A4O_X: Only 28 of the 253 domain residues found for /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/LYN/Inactive/3A4O_chainX.cif
Failed to create kinase for HASPIN_6G36_A: No valid atoms found for /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/HASPIN/Active/6G36_chainA.cif
